# The Armenian Name Inventor - a neural net learns to speak surname

**Instructor-led practical, chapter 11 (neural networks).** Builds on [41] (the network),
[42]-[43] (training it, early stopping), [44] (the Adam optimizer) and [46] (the same
machinery written from scratch in numpy).

Every family tree of ChatGPT leads back to the classic neural language model of
[Bengio et al., 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) - and that
model was nothing but an **embedding layer plus an MLP**. Exactly the parts we built in [41]
and trained in [42]-[43]. No attention, no recurrence, no magic.

Today we train that model on **689 Armenian surnames** scraped from the Armenian Wikipedia,
and then ask it to **invent surnames that do not exist**. Two tools are new:

- **PyTorch** - the deep-learning library the whole field runs on. First contact; every new
  piece gets explained the moment we use it.
- **Weights & Biases (wandb)** - an experiment tracker: a lab notebook that writes itself.
  Today involves ten trainings - six of them in one comparison - and nobody can hold ten
  loss curves in their head.

**The plan**

1. The data, 2. the alphabet, 3. context windows - and why the window is 3 letters
4. The model: an embedding (opened up: it is just a weight matrix) plus an MLP
5. Sampling: asking the net to write
6. How good is good? Loss, perplexity, and two baselines to beat
7. Training, logged to wandb
8. The payoff
9. Which knob matters? A small comparison
10. Saving the model and loading it back
11. - 14. Bonuses: temperature, memorization, a map of the letters, steering

**Runtime:** a full rerun takes ~6 min on the instructor's laptop CPU. Slow cells say so on
their first line; the big ones are the UMAP map in bonus 13 (~2-3 min, nearly all one-time
compilation), the wandb-logged trainings in sections 7 and 9 (~2 min together) and the
memorization run in bonus 12 (~70 s).

Seed is fixed to `509`. Everything runs on a laptop CPU.

In [1]:
# ~15 s (importing torch and wandb)
import copy
import os
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb

# install into the course venv if missing:  uv pip install torch wandb plotly umap-learn
SEED = 509
torch.manual_seed(SEED)

# This model is tiny (~8k weights). Splitting such small matrix products across CPU threads
# costs more than it saves: measured on the instructor laptop, one thread trains an epoch in
# 66 ms vs 130 ms with torch's default. For big models it is the opposite - keep the default.
torch.set_num_threads(1)

# plot colors (the Armenian flag, plus gray for reference lines)
RED, BLUE, ORANGE, GRAY = "#D90012", "#0033A0", "#F2A800", "#888888"

print("torch", torch.__version__, "| wandb", wandb.__version__)

torch 2.9.0+cpu | wandb 0.28.2


In [2]:
# Credentials hygiene: an API key never goes into a notebook (notebooks get committed,
# shared, screenshotted). Ours lives in the repo-root .env (gitignored) as
# WANDB_API_KEY=... - loaded here if present; otherwise wandb falls back to `wandb login`.
env_file = Path("../../.env")
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        key, _, value = line.partition("=")
        if key == "WANDB_API_KEY" and value.strip():
            os.environ["WANDB_API_KEY"] = value.strip()

print("WANDB_API_KEY is set:", "WANDB_API_KEY" in os.environ)

WANDB_API_KEY is set: True


## 1. The data: every Armenian surname Wikipedia knows

The Armenian Wikipedia keeps a category
[«Հայկական ազգանուններ»](https://hy.wikipedia.org/wiki/Կատեգորիա:Հայկական_ազգանուններ) with
one page per surname. The prep script `py_src/fetch_surnames.py` pulled every page title,
kept only titles made purely of Armenian letters (dropping disambiguation pages like
«Ազարյան (ազգանուն)» and hyphenated forms), lowercased them so capitals do not double our
alphabet, and pinned the result to `data/surnames_hy.txt` - so this notebook needs no
internet.

In [3]:
names = open("data/surnames_hy.txt", encoding="utf-8").read().split()

lengths = [len(n) for n in names]
n_yan = sum(n.endswith("յան") for n in names)
print(f"surnames: {len(names)}")
print(f"length min/mean/max: {min(lengths)}/{np.mean(lengths):.1f}/{max(lengths)}")
print(f"ending in -յան: {n_yan} ({n_yan / len(names):.0%})")
print("a few:", ", ".join(names[100:108]))

surnames: 689
length min/mean/max: 5/8.5/14
ending in -յան: 660 (96%)
a few: անանյան, անասյան, անդրանիկյան, անդրեասյան, անդրիասյան, անթառամյան, անջելյան, անտիկյան


## 2. The alphabet, discovered from the data

**Predict first:** how many *distinct characters* do 689 surnames use? (The Armenian
alphabet has 39 letters - do surnames use all of them?)

We do not hardcode the alphabet; we read it off the data. We also add one special character,
`.`, which does double duty:

- at the **start**, it pads the context: `...` means "nothing written yet";
- at the **end**, it is the answer "stop": this is how the model will learn that «-յան» is a
  way to finish. Without an end token the net could never end a name.

The model never confuses the two roles, because the context tells them apart: `...` comes
before the first letter, `յան` comes before the stop.

In [4]:
chars = sorted(set("".join(names)))
stoi = {c: i + 1 for i, c in enumerate(chars)}   # letter -> integer code (row number)
stoi["."] = 0                                    # the start/end token gets code 0
itos = {i: c for c, i in stoi.items()}           # integer code -> letter
V = len(stoi)

print("alphabet found in the data:", "".join(chars))
print(f"letters found: {len(chars)}, vocabulary size V (letters + '.'): {V}")

armenian = {chr(c) for c in range(ord("ա"), ord("և") + 1)}   # all lowercase Armenian letters
print("Armenian letters that never occur in a surname:", sorted(armenian - set(chars)))

alphabet found in the data: աբգդեզէթժիլխծկհձղճմյնշոչպջռսվտրցւփքօֆև
letters found: 38, vocabulary size V (letters + '.'): 39
Armenian letters that never occur in a surname: ['ը']


We hold out 20% of the **names** for validation. By names, not by training windows - windows
cut from the same name are near-duplicates, and splitting them across train/val would leak
(the same trap as ch2's rent model and the [06] evaluation lecture).

In [5]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(names))
n_tr = int(0.8 * len(names))
tr_names = [names[i] for i in perm[:n_tr]]
va_names = [names[i] for i in perm[n_tr:]]
tr_set, va_set = set(tr_names), set(va_names)   # for "is this a real name?" checks later
print(f"train names: {len(tr_names)}, val names: {len(va_names)}")

train names: 551, val names: 138


## 3. From characters to model food

**The trap first.** The obvious encoding - Ա=1, Բ=2, ... - lies to the model: it says Բ is
"twice" Ա and that Դ is "between" Գ and Ե. None of that is real. Characters are
**categories**, and we learned the honest encoding for categories back in the
feature-engineering lecture: **one-hot** (all zeros, a single 1). We will store integer
*codes*, but only as row numbers into one-hots - never as quantities (section 4 shows
exactly how).

**The task shape.** The model reads a **context window** of the previous $K$ characters and
predicts the next one. Sliding that window over «պողոսյան» with $K = 3$:

```
. . .  ->  պ          ո ս յ  ->  ա
. . պ  ->  ո          ս յ ա  ->  ն
. պ ո  ->  ղ          յ ա ն  ->  .   <- "now stop"
պ ո ղ  ->  ո  ...
```

One name of length $L$ yields $L{+}1$ training examples. The whole dataset is just these
(context, next-char) pairs.

In [6]:
def build_dataset(name_list, K):
    """Slide a K-wide window over every name; return (contexts X, next-chars Y)."""
    X, Y = [], []
    for n in name_list:
        ctx = [0] * K                      # start: K end-tokens as padding
        for ch in n + ".":                 # append '.' so the net learns to stop
            X.append(ctx)
            Y.append(stoi[ch])
            ctx = ctx[1:] + [stoi[ch]]     # slide the window one letter to the right
    return torch.tensor(X), torch.tensor(Y)


def decode(row):
    """Integer codes back to letters, e.g. tensor([25, 23, 17]) -> 'պող'."""
    return "".join(itos[int(i)] for i in row)

**Two minutes on tensors,** exactly where we first need them: a `torch.tensor` is a numpy
array that (a) remembers how it was computed, so gradients can flow backward through it
(that is autograd - the machinery running [42]'s backprop for us), and (b) can live on a GPU.
The two things you always check are `.shape` and `.dtype`.

In [7]:
K = 3
Xtr, Ytr = build_dataset(tr_names, K)
Xva, Yva = build_dataset(va_names, K)
print(f"train windows: {tuple(Xtr.shape)}, targets: {tuple(Ytr.shape)}, dtype: {Xtr.dtype}")
print(f"val windows:   {tuple(Xva.shape)}")

first = tr_names[0]
print(f"\nthe {len(first) + 1} windows cut from the first training name, «{first}»:")
for i in range(len(first) + 1):
    print(f"  '{decode(Xtr[i])}' -> '{itos[int(Ytr[i])]}'")

train windows: (5208, 3), targets: (5208,), dtype: torch.int64
val windows:   (1322, 3)

the 9 windows cut from the first training name, «մեխակյան»:
  '...' -> 'մ'
  '..մ' -> 'ե'
  '.մե' -> 'խ'
  'մեխ' -> 'ա'
  'եխա' -> 'կ'
  'խակ' -> 'յ'
  'ակյ' -> 'ա'
  'կյա' -> 'ն'
  'յան' -> '.'


### Why K = 3?

$K$ is the model's memory: how many previous letters it may look at. Two forces pull in
opposite directions, and both can be measured before training anything.

**Evidence 1 - what does the model need to see?** The most important decision in a surname
is *when to stop*. Among all training windows, how often is the next character `.`, given
the last 1, 2 or 3 letters?

In [8]:
def next_chars_after(context):
    """All characters that follow `context` in the training names (as integer codes)."""
    Xk, Yk = build_dataset(tr_names, len(context))
    code = torch.tensor([stoi[c] for c in context])
    matches = (Xk == code).all(dim=1)          # windows whose last len(context) letters match
    return Yk[matches]


for context in ["ն", "ան", "յան"]:
    nxt = next_chars_after(context)
    p_stop = (nxt == 0).float().mean().item()
    print(f"after '{context}' (K={len(context)}): seen {len(nxt)} times, "
          f"followed by '.' (stop) {p_stop:.0%} of the time")

after 'ն' (K=1): seen 716 times, followed by '.' (stop) 74% of the time


after 'ան' (K=2): seen 638 times, followed by '.' (stop) 83% of the time
after 'յան' (K=3): seen 534 times, followed by '.' (stop) 99% of the time


**Evidence 2 - what does a longer window cost?** Every extra letter multiplies the number of
possible contexts by $V = 39$. Our data does not grow. How many validation windows show a
context that never once appeared in training?

In [9]:
print(f"{'K':>2} {'possible contexts':>18} {'seen in training':>17} {'val windows, unseen context':>28}")
for k in [1, 2, 3, 5]:
    Xk_tr, _ = build_dataset(tr_names, k)
    Xk_va, _ = build_dataset(va_names, k)
    seen = {tuple(row) for row in Xk_tr.tolist()}
    unseen = np.mean([tuple(row) not in seen for row in Xk_va.tolist()])
    print(f"{k:>2} {V ** k:>18,} {len(seen):>17,} {unseen:>28.1%}")

 K  possible contexts  seen in training  val windows, unseen context
 1                 39                39                         0.0%
 2              1,521               422                         3.6%
 3             59,319             1,313                        16.5%
 5         90,224,199             2,730                        44.7%


Read the two printouts together:

- **K = 3 is the first window that sees the whole suffix.** Knowing only the last letter
  («ն»), stopping is right 74% of the time - plenty of names have ն mid-word. «ան» still
  appears inside stems (the name «անանյան» contains it three times, twice mid-word): 83%.
  «յան» makes the stop a near certainty: 99%. The single most common pattern in the data
  needs exactly three letters to recognize.
- **Every extra letter multiplies the possible contexts by 39, and 551 names cannot fill
  them.** At K = 3, one validation window in six already shows a context never seen in
  training; at K = 5 almost half do. A counting model is helpless on an unseen context. The
  MLP copes better - similar letters get similar vectors - but a longer window mostly gives
  it more to memorize.

So K = 3 is the sweet spot at 689 names: the smallest window that captures the suffix,
before the contexts go sparse. It is also the context Karpathy's *makemore* uses on 32,000
English names - the English sibling of this practical. Section 9 tests the claim with real
training runs at K = 1, 2, 3 and 5.

## 4. The model: an embedding and an MLP

### 4.1 The embedding layer is just a weight matrix

Start from what we know: a letter as a one-hot vector. Feed a one-hot into a linear layer,
i.e. multiply it by a weight matrix $W$ (one row per letter). The zeros wipe out every row of
$W$ except one:

$$\text{one-hot}(i) \cdot W = 0 \cdot W_0 + \dots + 1 \cdot W_i + \dots + 0 \cdot W_{V-1} = W_i
\quad \text{(row } i \text{ of } W\text{)}$$

So "multiply the one-hot by $W$" and "look up row $i$ of $W$" are **the same operation**; the
lookup just skips multiplying by all those zeros. That lookup table is what `nn.Embedding`
is. First with numbers small enough to check by eye - a 4-letter alphabet, 2 numbers per
letter:

In [10]:
toy_letters = ["ա", "բ", "գ", "դ"]
W = torch.tensor([[ 0.1,  0.2],     # row 0: the vector for ա
                  [ 1.0, -1.0],     # row 1: բ
                  [ 3.0,  4.0],     # row 2: գ
                  [-5.0,  0.5]])    # row 3: դ

onehot_g = torch.tensor([0.0, 0.0, 1.0, 0.0])          # one-hot for գ (position 2)
print("one-hot for գ:            ", onehot_g.tolist())
print("one-hot @ W  (multiply):  ", (onehot_g @ W).tolist())
print("W[2]         (look up):   ", W[2].tolist())

one-hot for գ:             [0.0, 0.0, 1.0, 0.0]


one-hot @ W  (multiply):   [3.0, 4.0]
W[2]         (look up):    [3.0, 4.0]


Now the real one. `nn.Embedding(V, d)` holds a single matrix of shape $(V, d)$: 39 rows, one
per character, $d$ numbers each. It is a trainable weight like any other - backprop updates
it - and its lookup is exactly the one-hot multiply:

In [11]:
emb = nn.Embedding(V, 8)                               # the lookup table: V rows x 8 columns
print("emb.weight shape:", tuple(emb.weight.shape), "| trainable:", emb.weight.requires_grad)

codes = torch.tensor([stoi[c] for c in "պող"])        # 3 letters -> 3 row numbers
print("codes for 'պող':", codes.tolist())

by_lookup = emb(codes)                                 # pick 3 rows           -> (3, 8)
onehots = F.one_hot(codes, num_classes=V).float()      # 3 one-hots             -> (3, 39)
by_matmul = onehots @ emb.weight                       # (3, 39) @ (39, 8)      -> (3, 8)
print("lookup == one-hot @ weight:", torch.allclose(by_lookup, by_matmul))

emb.weight shape:

 (39, 8) | trainable: True
codes for 'պող': [25, 23, 17]
lookup == one-hot @ weight: True


In [12]:
# Training touches only the rows that were used: the zeros in the one-hot send
# zero gradient to every other row.
by_lookup.sum().backward()
used_rows = (emb.weight.grad.abs().sum(dim=1) > 0).nonzero().flatten()
print("rows with a nonzero gradient:", used_rows.tolist(), "=", [itos[int(i)] for i in used_rows])
print(f"rows left untouched: {V - len(used_rows)} of {V}")

rows with a nonzero gradient:

 [17, 23, 25] = ['ղ', 'ո', 'պ']
rows left untouched: 36 of 39


**Why an explicit embedding layer**, when a plain MLP fed raw one-hots would also learn a
vector per letter (the rows of its first weight matrix)?

1. **Sharing across positions** - one vector per character, reused wherever it sits in the
   window; the raw version learns a separate copy of every letter for each of the $K$
   positions.
2. **A bottleneck you choose** - $Vd + Kdh$ weights instead of $KVh$ (here ~3.4k vs ~15k):
   characters are forced to share statistical strength, which matters at 689 names.
3. **An inspectable object** - "the vector for ա" is one row you can look at (we map exactly
   this table in bonus 13).

### 4.2 The architecture

Follow one context, «պող», through the network (numbers are tensor shapes, batch of 1):

```
  input         պ        ո        ղ              3 letters of context (K = 3)
                |        |        |
  codes         25       23       17             row numbers, not quantities
                |        |        |
  embedding    C[25]    C[23]    C[17]           3 rows of the table C (39 x 8), 8 numbers each
                \        |        /
  concatenate   [ 8 numbers | 8 numbers | 8 numbers ]  ->  24 numbers
                         |
  hidden        ReLU(W1 x + b1)                  24 -> 128 numbers
                         |
  output        W2 h + b2                        128 -> 39 scores ("logits")
                         |
  softmax       39 probabilities, summing to 1:  P(next = ա), P(next = բ), ..., P(next = .)
```

The same thing in PyTorch - `nn.Module` is the standard container: declare the layers in
`__init__`, say how data flows through them in `forward`:

In [13]:
class NameMLP(nn.Module):
    def __init__(self, V, K, d, h):
        super().__init__()
        self.emb = nn.Embedding(V, d)      # V rows, one d-dim vector per character
        self.fc1 = nn.Linear(K * d, h)     # affine ([41]: weighted sum + bias) ...
        self.fc2 = nn.Linear(h, V)         # ... -> ReLU -> affine -> V logits

    def forward(self, x):                  # x: (batch, K) integer codes
        e = self.emb(x)                    # (batch, K, d)  look up K vectors
        e = e.view(x.shape[0], -1)         # (batch, K*d)   concatenate them
        h = torch.relu(self.fc1(e))        # (batch, h)     hidden layer
        return self.fc2(h)                 # (batch, V)     one score per possible next char

In [14]:
# Walk «պող» through a fresh model step by step, printing the shape at each stage.
model = NameMLP(V, K, d=8, h=128)

x = torch.tensor([[stoi[c] for c in "պող"]])       # a batch of 1 context
e = model.emb(x)
e_flat = e.view(1, -1)
hidden = torch.relu(model.fc1(e_flat))
logits = model.fc2(hidden)
probs = F.softmax(logits, dim=1)

for label, t in [("codes", x), ("embeddings", e), ("concatenated", e_flat),
                 ("hidden (after ReLU)", hidden), ("logits", logits), ("probabilities", probs)]:
    print(f"{label:>20}: shape {tuple(t.shape)}")
print(f"{'':>20}  probabilities sum to {probs.sum().item():.3f}")

               codes: shape (1, 3)


          embeddings: shape (1, 3, 8)
        concatenated: shape (1, 24)
 hidden (after ReLU): shape (1, 128)
              logits: shape (1, 39)
       probabilities: shape (1, 39)
                      probabilities sum to 1.000


### 4.3 How many parameters?

Count layer by layer: the embedding table, then weights + biases of each linear layer.

$$\underbrace{V d}_{\text{embedding}} \;+\; \underbrace{(K d)\, h + h}_{\text{hidden layer}}
\;+\; \underbrace{h V + V}_{\text{output layer}}
= 39 \cdot 8 + 24 \cdot 128 + 128 + 128 \cdot 39 + 39 = 8{,}543$$

PyTorch agrees (note that it stores a `Linear` weight as (out, in), so `fc1.weight` prints
as (128, 24)):

In [15]:
for name, p in model.named_parameters():
    print(f"{name:>11}: shape {str(tuple(p.shape)):>9} -> {p.numel():>5} numbers")

d, h = 8, 128
by_formula = V * d + (K * d) * h + h + h * V + V
print(f"\ntotal: {sum(p.numel() for p in model.parameters())}   (formula: {by_formula})")

 emb.weight: shape   (39, 8) ->   312 numbers
 fc1.weight: shape (128, 24) ->  3072 numbers
   fc1.bias: shape    (128,) ->   128 numbers
 fc2.weight: shape (39, 128) ->  4992 numbers
   fc2.bias: shape     (39,) ->    39 numbers

total: 8543   (formula: 8543)


### 4.4 The untrained net has an opinion

Before any training, feed it the context «պող» and look at its next-character
probabilities. **Predict first:** what *should* a net with random weights believe?

In [16]:
ctx = torch.tensor([[stoi[c] for c in "պող"]])
with torch.no_grad():                       # predicting, not learning -> no gradient tape
    probs = F.softmax(model(ctx), dim=1)[0]

top = probs.argsort(descending=True)[:5]
print("untrained P(next char | 'պող'), top 5:")
for i in top:
    print(f"  '{itos[int(i)]}': {probs[i]:.3f}")
print(f"\n(compare: uniform would be 1/{V} = {1 / V:.3f} - random weights are near-clueless)")

untrained P(next char | 'պող'), top 5:
  'զ': 0.046
  'թ': 0.043
  'ջ': 0.039
  'ց': 0.038
  'բ': 0.036

(compare: uniform would be 1/39 = 0.026 - random weights are near-clueless)


## 5. Sampling: asking the net to write

Generation is just the forward pass in a loop: start from an all-`.` context, sample the
next character from the softmax (`torch.multinomial` draws proportionally to probability -
we do not always take the most likely letter, or every name would be the same), slide the
window, and stop when the net emits `.`.

The `temp` argument is a knob we explain in bonus 11; leave it at 1 until then. We sample
from the **untrained** net first - this is the "before" picture. (An empty entry in the list
means the net drew `.` as its very first character: a zero-letter name.)

In [17]:
@torch.no_grad()                                # generating, not learning: no gradient tape
def sample_names(model, K, n=20, temp=1.0, seed=SEED):
    gen = torch.Generator().manual_seed(seed)   # seeded -> same names on every rerun
    out = []
    for _ in range(n):
        ctx, s = [0] * K, ""                    # start: K end-tokens = "nothing written yet"
        while True:
            logits = model(torch.tensor([ctx]))[0]
            if temp == 0:                       # T = 0: always the single most likely char
                i = int(logits.argmax())
            else:
                p = F.softmax(logits / temp, dim=0)
                i = int(torch.multinomial(p, 1, generator=gen))   # draw by probability
            if i == 0 or len(s) > 25:           # '.' = "done" (25 letters = safety cap)
                break
            s += itos[i]
            ctx = ctx[1:] + [i]                 # slide the window
        out.append(s)
    return out


print("the UNTRAINED net writes:")
print(", ".join(n.capitalize() for n in sample_names(model, K, n=12)))

the UNTRAINED net writes:


Ժֆգզբխքհէևլւբծէւսղյշգխզգծգ, Խմմցյրծէզջկչեսնցքչխգմևքցգֆ, Սպւձպֆյձիբչէւփօվիէէչեա, Զռփբչթիգէփբւ, , Ծֆքչտշիգօկթջհցւթնզլթջ, Յնչքյսօրչվզքղիտխւհտհղշջձցշ, Ռթնխ, Սժգսոօվգչշքեյդվիածձթգջբջչտ, Ցլսմչօաշձզմխդսմվբյռտփկցխհշ, Զձօհէչթվյաեթդնկժչվփմյկբշձխ, Իտեծջչզթսղմֆտբֆչճդելիռճչիբ

## 6. How good is good? Loss, perplexity, and two baselines

Before training anything, decide how to score a model - and what score "no skill" gets.

### 6.1 The loss: average surprise

For every window the model outputs 39 probabilities. We look only at the probability $p$ it
gave to the character that **actually came next**, and charge $-\log p$ - the model's
*surprise*. The loss is the average surprise over all windows. This is **cross-entropy**,
the same log-loss as [11], now over 39 classes instead of 2.

In [18]:
for p in [1.0, 0.5, 0.2, 1 / V, 0.01]:
    print(f"p(correct char) = {p:.3f}  ->  surprise -log p = {np.log(1 / p):.2f}")

p(correct char) = 1.000  ->  surprise -log p = 0.00
p(correct char) = 0.500  ->  surprise -log p = 0.69
p(correct char) = 0.200  ->  surprise -log p = 1.61
p(correct char) = 0.026  ->  surprise -log p = 3.66
p(correct char) = 0.010  ->  surprise -log p = 4.61


Certain and right costs 0; a coin flip costs 0.69; the smaller $p$, the steeper the charge.
Being confidently wrong ($p = 0.01$) costs 4.6 - one such mistake outweighs many good guesses.

### 6.2 Baseline 1: uniform guessing

A model that knows nothing gives every character the same $1/V$. Its surprise is
$-\log(1/V) = \log V$ on every single window, so that is its loss. Every curve we draw today
is read against this line.

In [19]:
uniform_loss = float(np.log(V))
print(f"uniform guessing: every character gets 1/{V}  ->  loss = log({V}) = {uniform_loss:.3f}")

uniform guessing: every character gets 1/39  ->  loss = log(39) = 3.664


### 6.3 Perplexity: the loss in human units

"Loss 3.66" is hard to feel. **Perplexity** $= e^{\text{loss}}$ undoes the log and answers:
*among how many equally likely letters is the model effectively choosing?* A model that
always narrows the next letter down to $k$ equally likely candidates (one of them right) has
loss $\log k$ - so its perplexity is exactly $k$:

In [20]:
for k in [V, 7, 2, 1]:
    print(f"torn between {k:>2} letters: loss log({k}) = {np.log(k):.3f}  ->  "
          f"perplexity e^{np.log(k):.3f} = {np.exp(np.log(k)):.0f}")

torn between 39 letters: loss log(39) = 3.664  ->  perplexity e^3.664 = 39
torn between  7 letters: loss log(7) = 1.946  ->  perplexity e^1.946 = 7
torn between  2 letters: loss log(2) = 0.693  ->  perplexity e^0.693 = 2
torn between  1 letters: loss log(1) = 0.000  ->  perplexity e^0.000 = 1


Uniform guessing: perplexity 39, the whole alphabet. A perfect model: perplexity 1. Every
model today lands somewhere in between.

### 6.4 Baseline 2: the bigram - counting letter pairs

The cheapest model with any memory: **count**, over the training names, how often each
character follows each character. That is a $V \times V$ table. It knows exactly one thing -
the previous letter - so it is the counting version of a $K = 1$ model.

In [21]:
counts = np.zeros((V, V))                  # counts[a, b] = how often b follows a
for n in tr_names:
    seq = "." + n + "."                    # '.' marks both the start and the end
    for a, b in zip(seq, seq[1:]):
        counts[stoi[a], stoi[b]] += 1

row = counts[stoi["յ"]]
print(f"'յ' occurs {int(row.sum())} times in the training names; what comes next:")
for j in np.argsort(-row)[:5]:
    print(f"  յ -> {itos[j]}: {int(row[j]):>3} times  ({row[j] / row.sum():.1%})")

'յ' occurs 569 times in the training names; what comes next:
  յ -> ա: 542 times  (95.3%)
  յ -> ո:  10 times  (1.8%)
  յ -> ր:   5 times  (0.9%)
  յ -> ե:   3 times  (0.5%)
  յ -> վ:   2 times  (0.4%)


In [22]:
letters = [itos[i] for i in range(V)]
fig = go.Figure(go.Heatmap(
    z=np.log1p(counts), x=letters, y=letters, customdata=counts,
    colorscale=[[0, "white"], [1, BLUE]], colorbar=dict(title="log(1+count)"),
    hovertemplate="%{y} -> %{x}: %{customdata:.0f} times<extra></extra>"))
fig.update_layout(title="Bigram counts: row = this letter, column = the letter after it",
                  xaxis=dict(title="next letter", dtick=1),
                  yaxis=dict(title="this letter", dtick=1, autorange="reversed"),
                  width=720, height=700, template="plotly_white")
fig.show()

Hover over the cells. The «յ» row is nearly one dark cell (յ -> ա, the «-յան» suffix again),
the «.» row shows which letters start names, the «.» column which letters end them - and
most of the table is white: pairs that never occur.

**From counts to probabilities:** divide each row by its total. But the white cells are a
trap: a pair never seen in training gets probability 0, and $-\log 0 = \infty$. A single
unseen pair in the validation names makes the average loss infinite. How many are there?

In [23]:
def bigram_val_loss(P):
    """Average surprise of the probability table P on every letter pair of the val names."""
    surprises = [-np.log(P[stoi[a], stoi[b]])
                 for n in va_names for a, b in zip("." + n, n + ".")]
    return float(np.mean(surprises)), surprises


print(f"table cells never seen in training: {int((counts == 0).sum())} of {V * V}")

P_raw = counts / counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore"):         # log(0) = -inf is exactly the point here
    raw_loss, raw_surprises = bigram_val_loss(P_raw)
print(f"validation pairs never seen in training: {int(np.isinf(raw_surprises).sum())} "
      f"of {len(raw_surprises)}")
print(f"raw bigram val loss: {raw_loss}")

table cells never seen in training: 1096 of 1521
validation pairs never seen in training: 48 of 1322
raw bigram val loss: inf


The standard fix is **add-one (Laplace) smoothing**: pretend every pair was seen once more
than it was. Unseen pairs get a small nonzero probability; the price is that frequent pairs
give a little probability away:

In [24]:
P_bigram = (counts + 1) / (counts + 1).sum(axis=1, keepdims=True)   # add-one smoothing
a, b = stoi["յ"], stoi["ա"]
print(f"P(ա | յ): raw {P_raw[a, b]:.1%} -> smoothed {P_bigram[a, b]:.1%}")

bigram_loss, _ = bigram_val_loss(P_bigram)
print(f"\nuniform: val loss {uniform_loss:.3f}   perplexity {np.exp(uniform_loss):.1f}")
print(f"bigram:  val loss {bigram_loss:.3f}   perplexity {np.exp(bigram_loss):.1f}")

P(ա | յ): raw 95.3% -> smoothed 89.3%

uniform: val loss 3.664   perplexity 39.0
bigram:  val loss 1.948   perplexity 7.0


In [25]:
# The bigram can write, too: start at '.', draw the next letter from the current letter's row.
g_big = np.random.default_rng(SEED)
big_samples = []
for _ in range(10):
    i, s = 0, ""
    while True:
        i = g_big.choice(V, p=P_bigram[i])
        if i == 0 or len(s) > 25:
            break
        s += itos[i]
    big_samples.append(s.capitalize())
print("the bigram model writes:", ", ".join(big_samples))

the bigram model writes: Գւն, Փալբան, Դյապճժրռգրոյակպեցևջաշմադալ, Թեյաջղհանարիվան, Պան, Ան, Եւծկյիրջռին, Բդան, Թևզյագյամասիծշդծիգկունջանի, Վալյարյարուհմյան


One letter of memory buys a lot: from perplexity 39 to about 7. Some outputs are almost
names - but nothing stops a bigram from rambling for 26 characters, because it has no idea
how much name has already happened. That is exactly the job of a longer context window.

So the MLP has to beat **the bigram**, not the clueless 3.66. (A $K = 1$ network learns
essentially this table by gradient descent - [11]'s softmax regression rediscovering the
counts. Section 9 checks that.)

## 7. Training: the loop, for real

This is the standard PyTorch training loop - the one in your ch11 homework, with [42]'s
`loss.backward()` at its core:

```python
for epoch in range(n_epochs):
    for xb, yb in batches:           # mini-batches
        optimizer.zero_grad()        # reset grads
        out = model(xb)              # forward pass
        loss = loss_fn(out, yb)
        loss.backward()              # backprop (autograd runs [42]'s delta recursion)
        optimizer.step()             # update weights
```

- **An epoch** is one pass over all 5,208 training windows, shuffled, in **mini-batches of
  1,024** - so 6 weight updates per epoch.
- **The optimizer is Adam** ([44]): gradient descent with momentum and a per-weight step
  size - the default choice for networks like this one.
- **The loss** is `F.cross_entropy`, which takes the raw **logits**, not softmax outputs: it
  applies log-softmax internally because that is numerically safer. Same math, better
  arithmetic.
- **Early stopping** ([43]) is built in from the start: with 689 names this model overfits
  *fast* - the validation loss bottoms out within the first few dozen epochs and then climbs.
  So `fit()` tracks the best-validation epoch and **keeps those weights**.

In [26]:
def evaluate(model, X, Y):
    """The loss (average cross-entropy) of `model` on windows X with targets Y."""
    with torch.no_grad():
        return float(F.cross_entropy(model(X), Y))


def fit(config, on_epoch=None, seed=SEED):
    """Train a NameMLP with mini-batch Adam and [43]'s early stopping.
    Returns (best-validation model, loss history, best val loss, best epoch).
    Pure training - pass on_epoch(ep, model, train_loss, val_loss, best_val) to watch it."""
    torch.manual_seed(seed)                          # controls the initial weights
    K = config["K"]
    Xtr, Ytr = build_dataset(tr_names, K)
    Xva, Yva = build_dataset(va_names, K)
    model = NameMLP(V, K, config["d"], config["h"])
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    g = torch.Generator().manual_seed(seed)          # controls the batch shuffling
    hist = {"train": [], "val": []}
    best_val, best_ep, best_state = float("inf"), 0, None

    for ep in range(1, config["epochs"] + 1):
        # one epoch: a shuffled pass over all training windows, in mini-batches
        idx = torch.randperm(len(Xtr), generator=g)
        for k in range(0, len(idx), config["batch"]):
            b = idx[k:k + config["batch"]]
            loss = F.cross_entropy(model(Xtr[b]), Ytr[b])
            opt.zero_grad()
            loss.backward()
            opt.step()

        tl, vl = evaluate(model, Xtr, Ytr), evaluate(model, Xva, Yva)
        hist["train"].append(tl)
        hist["val"].append(vl)
        if vl < best_val:                            # early stopping, [43]:
            best_val, best_ep = vl, ep               # remember the best epoch's weights
            best_state = copy.deepcopy(model.state_dict())
        if on_epoch:
            on_epoch(ep, model, tl, vl, best_val)

    model.load_state_dict(best_state)                # restore the val-minimum model
    return model, hist, best_val, best_ep

### Weights & Biases in one paragraph

An experiment tracker records, for every **run**: the exact **config** (every knob), the
**metrics** over time (our losses), and any artifacts you log (we will log the *names the
net writes* every few epochs). You then compare runs in a dashboard instead of in your
memory. The discipline it enforces - *no experiment without its settings recorded* - is the
actual lesson; by hand this dies at run 3.

Setup: create a free account at [wandb.ai](https://wandb.ai), copy your API key from
[wandb.ai/authorize](https://wandb.ai/authorize) into the repo-root `.env` as
`WANDB_API_KEY=...` (the setup cell above already loads it) - or run `wandb.login()` once.
**No account / no internet right now?** Uncomment the `WANDB_MODE = "offline"` line at
the top of the next cell: everything still runs and logs to a local folder, nothing is
uploaded - nobody is blocked.

One habit worth keeping forever: before a batch of runs, **check which account will
receive them** - the next cell prints it before anything trains (or confirms that you
are offline). Shared machines accumulate stale logins; runs that land in the wrong
account are annoying to chase.

In [27]:
# ~4 s (asks the wandb server who you are)
# No wandb account, or no internet? Uncomment the next line: runs are then written to a
# local ./wandb folder instead of uploaded, and nothing else in the notebook changes.
# os.environ["WANDB_MODE"] = "offline"

if os.environ.get("WANDB_MODE") == "offline":
    print("WANDB_MODE=offline: runs are logged to ./wandb, nothing is uploaded")
else:
    # Which account gets these runs? A stored login proves *a* login exists, not whose.
    # (An error here means no login at all: run wandb.login(), or go offline above.)
    print("runs will go to wandb entity:", wandb.Api().default_entity)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


runs will go to wandb entity: tarkhanyan


In [28]:
def cfg_label(c):
    """A readable run name, e.g. 'K3-d8-h128-lr0.003'."""
    return f"K{c['K']}-d{c['d']}-h{c['h']}-lr{c['lr']}"


def train_logged(config, prefix="main"):
    """fit() + one wandb run that logs the losses every epoch and the names the net writes."""
    K = config["K"]
    # silent=True: none of wandb's own upload chatter in the notebook - we print the link ourselves
    run = wandb.init(project="armenian-name-inventor", name=f"{prefix}-{cfg_label(config)}",
                     config=config, settings=wandb.Settings(silent=True))
    print("wandb run:", run.url or f"offline, saved under {run.dir}")

    def on_epoch(ep, model, tl, vl, best):
        row = {"train_loss": tl, "val_loss": vl, "best_val_loss": best,
               "baseline_uniform": uniform_loss, "baseline_bigram": bigram_loss}
        # Log what the net writes right now. Log-spaced early epochs matter most:
        # the dramatic learning (gibberish -> '-yan') happens in the first ~16 epochs,
        # because the suffix is the single most frequent pattern in the data.
        if ep in (1, 2, 4, 8, 16) or ep % config["sample_every"] == 0:
            tbl = wandb.Table(columns=["name"])
            for s in sample_names(model, K, n=10, seed=SEED + ep):
                tbl.add_data(s.capitalize())
            row["samples"] = tbl
        wandb.log(row, step=ep)

    model, hist, best_val, best_ep = fit(config, on_epoch)
    run.finish()
    print(f"best val {best_val:.3f} at epoch {best_ep}/{config['epochs']} "
          f"(uniform {uniform_loss:.3f}, bigram {bigram_loss:.3f}); kept those weights")
    return model, hist, best_ep

In [29]:
# ~1 min: ~10 s of training, the rest is wandb starting up and uploading
config = dict(K=3, d=8, h=128, lr=3e-3, epochs=150, batch=1024, sample_every=25)
model, hist, best_ep = train_logged(config)

wandb run: https://wandb.ai/tarkhanyan/armenian-name-inventor/runs/xh59xwom


best val 1.673 at epoch 41/150 (uniform 3.664, bigram 1.948); kept those weights


The same story the dashboard tells, drawn right here:

In [30]:
epochs = list(range(1, config["epochs"] + 1))
fig = go.Figure()
fig.add_scatter(x=epochs, y=hist["train"], name="train loss", line=dict(color=BLUE, width=2))
fig.add_scatter(x=epochs, y=hist["val"], name="validation loss", line=dict(color=RED, width=2))
fig.add_hline(y=uniform_loss, line=dict(color=GRAY, dash="dash", width=1),
              annotation_text="uniform guessing", annotation_position="bottom right")
fig.add_hline(y=bigram_loss, line=dict(color=GRAY, dash="dot", width=1),
              annotation_text="bigram", annotation_position="bottom")
fig.add_vline(x=best_ep, line=dict(color=GRAY, dash="dash", width=1),
              annotation_text=f"best val: epoch {best_ep} (kept)", annotation_position="top right")
fig.update_layout(title="Train keeps falling, validation turns around: early stopping keeps the valley",
                  xaxis_title="epoch", yaxis_title="loss (average cross-entropy)",
                  width=820, height=440, template="plotly_white",
                  legend=dict(x=0.99, y=0.72, xanchor="right"))
fig.show()

## 8. The payoff

Same `sample_names` function as before - but the weights have changed. We ask for 200
inventions and look at the first 30. **Predict first:** what share will end in «-յան»?
(96% of the training names do.)

Then check two more things. How many "inventions" are **real training surnames** repeated
back (memorized - keep that number in mind, it becomes a whole section later)? And the best
kind of hit: how many are **real surnames the net never saw** - names that sit in the
held-out validation set? A model that produces one of those has learned what an Armenian
surname looks like, not just a list of them.

In [31]:
invented = sample_names(model, K, n=200)
print("the TRAINED net writes (the first 30 of 200):\n")
for i in range(0, 30, 5):
    print("  " + ", ".join(n.capitalize() for n in invented[i:i + 5]))

copied = [n for n in invented if n in tr_set]
rediscovered = [n for n in invented if n in va_set]
print("\nof all 200 inventions:")
print(f"  end in -յան:                                 {sum(n.endswith('յան') for n in invented)}")
print(f"  copied from the training names:              {len(copied)}")
print(f"  real surnames it never saw (validation set): {len(rediscovered)}  "
      f"{[n.capitalize() for n in rediscovered]}")
print(f"  distinct names:                              {len(set(invented))}")

the TRAINED net writes (the first 30 of 200):

  Ժիրզարյան, Լալյան, Աբագրյան, Ղամարյան, Զոդունն
  Ավայան, Մարֆան, Բոնյան, Երշեկյան, Իեյան
  Փբոյնյան, Բունց, Հաշիգնյան, Աբանյան, Իյան
  Յան, Բանյան, Մելիյան, Աբաջյան, Նսյան
  Սվգչշանյան, Ալելյան, Չիկյան, Աբակյան, Դոմյան
  Սոյան, Թազորյան, Սանյան, Կոսյանյաբիբյան, Երջանթարյանյան

of all 200 inventions:
  end in -յան:                                 190
  copied from the training names:              9
  real surnames it never saw (validation set): 1  ['Աբելյան']
  distinct names:                              191


190 of the 200 end in «-յան» - 95%, against 96% in the data: the suffix is learned at almost
exactly the rate the data teaches it. 9 are copied training names; hold on to that number.
And one invention, **Աբելյան**, is a real surname the net never saw - it sits in the
held-out validation set. Nobody showed the model that name; it produced it from what
Armenian surnames look like in general. That is generalization in its purest form.

Now the ladder. Read it honestly: the humble bigram already covered most of the distance from
clueless to name-shaped; the MLP's edge over it is real but modest, earned on the patterns
longer than one letter. At 689 names, much of "sounding Armenian" *is* letter-pair
statistics.

In [32]:
mlp_loss = evaluate(model, Xva, Yva)
ladder = {"uniform guessing": uniform_loss, "bigram (counts)": bigram_loss, "MLP (ours)": mlp_loss}
for name, loss in ladder.items():
    print(f"{name:>17}: val loss {loss:.3f}   perplexity {np.exp(loss):.1f}")

fig = go.Figure(go.Bar(
    x=[np.exp(v) for v in ladder.values()], y=list(ladder), orientation="h",
    marker_color=[GRAY, GRAY, RED],
    text=[f"perplexity {np.exp(v):.1f}   (loss {v:.2f})" for v in ladder.values()],
    textposition="outside", cliponaxis=False))
fig.update_layout(title="The ladder: among how many letters is each model effectively choosing?",
                  xaxis=dict(title="perplexity (lower is better)", range=[0, 55]),
                  yaxis=dict(autorange="reversed"),
                  width=780, height=300, template="plotly_white", margin=dict(r=60))
fig.show()

 uniform guessing: val loss 3.664   perplexity 39.0
  bigram (counts): val loss 1.948   perplexity 7.0
       MLP (ours): val loss 1.673   perplexity 5.3


### Read the dashboard

Open the run's wandb page (the link printed by the training cell). Three things to find:

1. **Charts -> `train_loss` / `val_loss`** against the two baselines. This is [43]'s
   read-your-curves frame happening live: train loss falls and keeps falling, while the
   validation curve bottoms out within the first few dozen epochs and then **turns upward**
   - the model is memorizing 551 surnames, not learning "Armenian". That early valley is
   exactly why `fit()` kept the best-epoch weights.
2. **The `samples` table**, with its step slider - the first checkpoints are log-spaced
   (epochs 1, 2, 4, 8, 16) because that is where the learning is dramatic. Notice *what*
   the net learns first: the «-յան» ending shows up almost immediately, long before the
   stems sound Armenian. No mystery - 96% of training names carry that suffix, so it is
   the single most loss-reducing pattern available, and gradient descent buys the cheapest
   improvement first. Frequent patterns are learned early; rare structure comes late (or
   never, at this data size).
3. **Overview -> Config** - the exact knobs of this run, recorded without you doing
   anything. That is the point of a tracker.

One more number worth savoring: this model has **more parameters (8,543) than training
windows (5,208)**. Classical statistics says that must end in tears, yet with early stopping
it generalizes fine. Modern deep learning lives almost entirely in this regime; you have just
met it at surname scale.

## 9. Which knob matters? A small comparison

Two questions. Does the context $K$ behave the way section 3 predicted? And does a bigger
network help at 689 names? This is **hyperparameter search** - lecture [08]'s machinery on a
new model. We vary **one knob at a time** around the base config:

- context $K \in \{1, 2, 3, 5\}$,
- hidden size $h \in \{32, 128, 512\}$,

which costs 6 trainings (the base config sits in both lists), and log all validation curves
into a **single wandb run** (`hp-comparison`): one chart per knob, curves labeled by value.

Two honest details:

- **The score is the best val loss reached during the run** - early stopping applied to the
  comparison itself. Final val loss would punish big models for overshooting their early
  minimum, not for being bad. The flip side: a run whose best epoch is at the very end of the
  budget was still improving, so its score is capped by the budget. The leaderboard flags it.
- **How big a difference is real?** We retrain the base config with two more seeds: the
  spread between identical configs is the noise floor. A gap smaller than that is not a
  finding.

At real scale you would not do this by hand: **wandb sweeps** automate random search ([08]:
beats grid for the same budget) across many machines. Same idea, more machinery.

In [33]:
# ~30 s: 6 trainings of 100 epochs
base = dict(K=3, d=8, h=128, lr=3e-3, epochs=100, batch=1024)
knobs = {"K": [1, 2, 3, 5], "h": [32, 128, 512]}

run = wandb.init(project="armenian-name-inventor", name="hp-comparison",
                 config={"base": base, "knobs": knobs}, settings=wandb.Settings(silent=True))
print("wandb run:", run.url or f"offline, saved under {run.dir}")

results = {}                               # cfg_label -> (config, history, best_val, best_ep)
t0 = time.time()
for knob, values in knobs.items():
    curves, labels = [], []
    for v in values:
        cfg = dict(base, **{knob: v})      # the base config with one knob changed
        key = cfg_label(cfg)
        already_trained = key in results   # the base config appears in both lists
        if not already_trained:
            _, hist_k, best_val_k, best_ep_k = fit(cfg)
            results[key] = (cfg, hist_k, best_val_k, best_ep_k)
        _, hist_k, best_val_k, best_ep_k = results[key]
        curves.append(hist_k["val"])
        labels.append(f"{knob}={v}")
        note = "(base config, trained above)" if already_trained else f"({time.time() - t0:.0f} s elapsed)"
        print(f"  {knob}={v}: best val {best_val_k:.3f} at epoch {best_ep_k}   {note}")
    run.log({f"varying {knob}": wandb.plot.line_series(
        xs=list(range(1, base["epochs"] + 1)), ys=curves, keys=labels,
        title=f"val loss vs epoch - varying {knob}", xname="epoch")})

wandb run: https://wandb.ai/tarkhanyan/armenian-name-inventor/runs/vox2ed18


  K=1: best val 1.904 at epoch 36   (4 s elapsed)


  K=2: best val 1.721 at epoch 51   (8 s elapsed)


  K=3: best val 1.673 at epoch 41   (13 s elapsed)


  K=5: best val 1.673 at epoch 40   (19 s elapsed)


  h=32: best val 1.681 at epoch 99   (23 s elapsed)
  h=128: best val 1.673 at epoch 41   (base config, trained above)


  h=512: best val 1.707 at epoch 22   (34 s elapsed)


In [34]:
# ~15 s: 2 more trainings
# The noise floor: the SAME base config, two more seeds (other initial weights, other batch order)
seed_vals = [results[cfg_label(base)][2]]
for s in [SEED + 1, SEED + 2]:
    seed_vals.append(fit(base, seed=s)[2])
noise = max(seed_vals) - min(seed_vals)
print("base config, 3 seeds:", "  ".join(f"{v:.3f}" for v in seed_vals), f"  -> spread {noise:.3f}")

base config, 3 seeds: 1.673  1.672  1.674   -> spread 0.002


In [35]:
# ~5 s (wandb uploads the finished comparison run)
def n_params(cfg):
    return sum(p.numel() for p in NameMLP(V, cfg["K"], cfg["d"], cfg["h"]).parameters())


leaderboard = sorted(results.values(), key=lambda r: r[2])     # best val first
tbl = wandb.Table(columns=["config", "params", "best_val", "best_epoch"])
print(f"{'config':>22} {'params':>7} {'best val':>9} {'epoch':>6}")
for cfg, _, best_val_k, best_ep_k in leaderboard:
    flag = "  <- best epoch at the end of the budget: may still be improving" \
        if best_ep_k > 0.9 * cfg["epochs"] else ""
    print(f"{cfg_label(cfg):>22} {n_params(cfg):>7,} {best_val_k:>9.3f} {best_ep_k:>6}{flag}")
    tbl.add_data(cfg_label(cfg), n_params(cfg), round(best_val_k, 3), best_ep_k)
print(f"\nnoise floor (seed spread): {noise:.3f}     bigram baseline: {bigram_loss:.3f}")

run.log({"leaderboard": tbl})
run.finish()

                config  params  best val  epoch
    K5-d8-h128-lr0.003  10,591     1.673     40
    K3-d8-h128-lr0.003   8,543     1.673     41
     K3-d8-h32-lr0.003   2,399     1.681     99  <- best epoch at the end of the budget: may still be improving
    K3-d8-h512-lr0.003  33,119     1.707     22
    K2-d8-h128-lr0.003   7,519     1.721     51
    K1-d8-h128-lr0.003   6,495     1.904     36

noise floor (seed spread): 0.002     bigram baseline: 1.948


In [36]:
fig = go.Figure(go.Bar(
    x=[r[2] for r in leaderboard], y=[cfg_label(r[0]) for r in leaderboard], orientation="h",
    marker_color=[RED if r[0] == base else BLUE for r in leaderboard],
    text=[f"{r[2]:.3f}" for r in leaderboard], textposition="inside",
    insidetextanchor="end", textfont=dict(color="white")))
fig.add_vline(x=bigram_loss, line=dict(color=GRAY, dash="dot", width=1),
              annotation_text="bigram", annotation_position="top")
fig.update_layout(title=f"Best validation loss per config (red = base; seed noise {noise:.3f})",
                  xaxis=dict(title="best val loss (lower is better)", range=[0, 2.2]),
                  yaxis=dict(autorange="reversed"),
                  width=780, height=380, template="plotly_white", margin=dict(r=60))
fig.show()

Read the leaderboard against the noise floor (0.002):

- **K = 1 lands at 1.904, next to the bigram (1.948)** - the check promised in section 6.
  One letter of context, counted or learned, gets you to the same place; the network is a
  little better because add-one smoothing is a crude way to handle rare pairs.
- **K = 2 -> K = 3 is the last real step (1.721 -> 1.673)**, exactly where section 3 said it
  would be: three letters see the whole «-յան».
- **K = 5 ties K = 3** (1.673 vs 1.673, well inside the noise) with 24% more weights. More
  context bought nothing at 689 names.
- **h = 512 is worse (1.707)**: 4x the weights, it memorizes faster (best epoch 22) and lands
  higher. **h = 32** trails by 0.008, but it peaked at epoch 99 of 100 - still improving, so
  its score is capped by the epoch budget, not by the model.

The near-tie at the top *is* the finding: at 689 names none of these knobs is the
bottleneck. **The data is.** No architecture choice competes with "collect more names" -
which is, in one sentence, why the big-model era is above all a big-data era.

So we keep the base config (K = 3, d = 8, h = 128): the smallest model tied for first. The
model trained in section 7 already *is* that config, so there is nothing to retrain - it
becomes our final model. One honest caveat: val loss is a **proxy** for what we actually
want (charming names). Always eyeball a model's samples before believing its number.

## 10. Save the model, load it back

Training took seconds here; for a real model it takes weeks, and nobody retrains a model to
use it. So what has to go into the file? Three things - and forgetting either of the last two
is the classic mistake:

1. **The weights** - `model.state_dict()`, a dict of named tensors.
2. **The config** ($K$, $d$, $h$) - you must rebuild an empty model of the same shape before
   the weights can be poured into it.
3. **The vocabulary** (`stoi`) - the weights of "row 25" mean nothing unless row 25 still
   means պ. Rebuild the alphabet from a different name list and every row shifts.

We save the `state_dict`, not the model object: `torch.save(model)` pickles your class by its
import path, so the file breaks the day you rename or move the class. A `state_dict` is just
tensors.

In [37]:
ckpt_path = Path("data/name_inventor.pt")
checkpoint = {
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},   # CPU tensors load anywhere
    "config": config,
    "stoi": stoi,
}
torch.save(checkpoint, ckpt_path)

print(f"saved {ckpt_path} ({ckpt_path.stat().st_size / 1024:.0f} KB)")
print("inside:", list(checkpoint))
print("weights:", {k: tuple(v.shape) for k, v in checkpoint["state_dict"].items()})

saved data\name_inventor.pt (37 KB)
inside: ['state_dict', 'config', 'stoi']
weights: {'emb.weight': (39, 8), 'fc1.weight': (128, 24), 'fc1.bias': (128,), 'fc2.weight': (39, 128), 'fc2.bias': (39,)}


Now pretend it is next week and this is a fresh Python session. Loading is the save in
reverse: read the file, build an empty model from the config, pour the weights in. Two safety
details:

- `weights_only=True` (the default since PyTorch 2.6) refuses to run code hidden in the file.
  A `.pt` file is a pickle, and a pickle from the internet could otherwise execute anything
  on load.
- `load_state_dict` is **strict**: a missing or misshaped tensor raises an error instead of
  quietly loading a broken model.

In [38]:
ckpt = torch.load(ckpt_path, weights_only=True)
assert ckpt["stoi"] == stoi, "alphabet changed since saving - the rows would mean other letters"

cfg = ckpt["config"]
loaded = NameMLP(len(ckpt["stoi"]), cfg["K"], cfg["d"], cfg["h"])   # empty model, same shape
loaded.load_state_dict(ckpt["state_dict"])                          # strict: shapes must match
loaded.eval()        # inference mode - a no-op here, vital once a model has dropout/BatchNorm ([46])

print(f"val loss:  original {evaluate(model, Xva, Yva):.4f}   loaded {evaluate(loaded, Xva, Yva):.4f}")
same = sample_names(loaded, cfg["K"], n=10) == sample_names(model, K, n=10)
print("same 10 names from both:", same)

# From here on, the bonuses use the model we just loaded.
final_model, final_K = loaded, cfg["K"]

val loss:  original 1.6728   loaded 1.6728
same 10 names from both: True


## 11. Bonus: the temperature knob

Every chat app has a *temperature* setting. Here is all it is: divide the logits $z$ by a
number $T$ before the softmax.

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **$T = 1$** - the model's own distribution; everything we sampled so far.
- **$T < 1$** - the gaps between logits grow, so the favourite gets more and the long tail
  less: safer, more typical output.
- **$T > 1$** - the gaps shrink and the distribution flattens: wilder output, more nonsense.
  As $T \to \infty$ it becomes uniform - the model's knowledge is erased.
- **$T = 0$** - the limit of sharpening: all probability on the single top choice, no
  randomness left. This is **greedy decoding** (argmax). Dividing by 0 is undefined, so code
  treats $T = 0$ as argmax - exactly what `sample_names` does.

Three logits make it concrete:

In [39]:
z = torch.tensor([2.0, 1.0, 0.0])          # logits for three letters, say ա, բ, գ
print("logits:", z.tolist(), "\n")
print("T = 0    ->", F.one_hot(z.argmax(), 3).float().tolist(), "  (argmax: all on the top letter)")
for T in [0.5, 1.0, 2.0, 100.0]:
    p = F.softmax(z / T, dim=0)
    print(f"T = {T:<5} ->", [round(v, 3) for v in p.tolist()])

logits: [2.0, 1.0, 0.0] 

T = 0    -> [1.0, 0.0, 0.0]   (argmax: all on the top letter)
T = 0.5   -> [0.867, 0.117, 0.016]
T = 1.0   -> [0.665, 0.245, 0.09]
T = 2.0   -> [0.506, 0.307, 0.186]
T = 100.0 -> [0.337, 0.333, 0.33]


Now the real model. Which letter should a surname **start** with? That is one softmax - the
context `...` - shown at four temperatures (the 8 most likely first letters):

In [40]:
with torch.no_grad():
    z = final_model(torch.tensor([[0] * final_K]))[0]   # logits for the first letter
top = z.argsort(descending=True)[:8]                       # the 8 most likely first letters
temps = [0, 0.5, 1.0, 2.0]

fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"T = {T}" + (" (greedy)" if T == 0 else "") for T in temps])
for col, T in enumerate(temps, start=1):
    p = F.one_hot(z.argmax(), V).float() if T == 0 else F.softmax(z / T, dim=0)
    fig.add_bar(x=[itos[int(i)] for i in top], y=p[top].tolist(), marker_color=BLUE,
                text=[f"{v:.0%}" for v in p[top].tolist()], textposition="outside",
                cliponaxis=False, showlegend=False, row=1, col=col)
    print(f"T = {T}: the top 8 letters hold {p[top].sum():.0%} of the probability")
fig.update_yaxes(range=[0, 1.12], tickformat=".0%")
fig.update_yaxes(title_text="P(first letter)", row=1, col=1)
fig.update_layout(title="The same logits at four temperatures: sharper below 1, flatter above",
                  width=960, height=380, template="plotly_white",
                  uniformtext=dict(minsize=10, mode="show"))   # same label size on every bar
fig.show()

T = 0: the top 8 letters hold 100% of the probability
T = 0.5: the top 8 letters hold 95% of the probability
T = 1.0: the top 8 letters hold 73% of the probability
T = 2.0: the top 8 letters hold 48% of the probability


And the names themselves. We use a fresh seed, so the $T = 1$ row is new names rather than
a rerun of section 8's list, and 200 names per temperature so the counts mean something:

In [41]:
# ~6 s: 600 names
greedy = sample_names(final_model, final_K, n=5, temp=0)
print("T = 0 (greedy):", ", ".join(n.capitalize() for n in greedy))
print(f"   -> the same name every time. In the training names: {greedy[0] in tr_set}, "
      f"in the validation names: {greedy[0] in va_set}\n")

for T in [0.5, 1.0, 1.5]:
    s = sample_names(final_model, final_K, n=200, temp=T, seed=SEED + 1)
    print(f"T = {T}: copied from training {sum(x in tr_set for x in s):>2}/200 | "
          f"distinct {len(set(s)):>3}/200 | mean length {np.mean([len(x) for x in s]):.1f}")
    print("   " + ", ".join(n.capitalize() for n in s[:8]) + "\n")

T = 0 (greedy): Արանյան, Արանյան, Արանյան, Արանյան, Արանյան
   -> the same name every time. In the training names: False, in the validation names: False



T = 0.5: copied from training 23/200 | distinct 135/200 | mean length 7.3
   Պարյան, Արանյան, Բաբեկյան, Առան, Աբաջյան, Բալանյան, Ահետյան, Ասունյան



T = 1.0: copied from training  6/200 | distinct 195/200 | mean length 8.3
   Պավդալյան, Միրանյան, Մատյան, Ազունց, Դարաբյան, Թժշյան, Աբերդիշոսյան, Մանյանց



T = 1.5: copied from training  0/200 | distinct 199/200 | mean length 12.9
   Պժոդբեզյան, Հրավյան, Ոդտորաչալյան, Եդգրնբալրաթյշյան, Աբերդիշոսյան, Մանյմնցյան, Աղազյանյան, Տետսկելոնկբինզյանղիյան



Read the chart and the counts together:

- **T = 0** writes «Արանյան» every single time - the one most likely surname, and not a real
  one (it is in neither name list). Greedy decoding is for when you want *the* answer, the
  same every time; it can never surprise you.
- **T = 0.5** plays it safe: short names (7.3 letters on average), only 135 distinct out of
  200, and 23 copied from the training set - nearly four times as many as at T = 1. Low
  temperature amplifies memorization: the safest name is one the model has already seen.
- **T = 1.0** is the model's own distribution: 195 distinct names, 6 copies.
- **T = 1.5** gambles: every name is new, but the mean length jumps to 12.9 letters and the
  stems fall apart («Եդգրնբալրաթյշյան»).

Every generative product you have used ships this exact knob. Now you know it is one
division.

## 12. Bonus: overfitting is memorization

**Predict first:** we train the same model **8x longer** than section 7, with no early
stopping. Better names?

Every 100 epochs we measure two things: the val loss, and the share of 50 freshly generated
names that are **verbatim training names**. One subtlety: even a good model produces *some*
real names by chance (short, high-probability ones) - so the verdict is not "any real name =
guilty" but the **trajectory**: does the copied share climb while the val loss goes the
wrong way?

In [42]:
# ~70 s: 1200 epochs, with a progress line every 100
# A raw loop on purpose - NO early stopping here. We *want* to watch it overtrain.
torch.manual_seed(SEED)
K = 3
Xtr, Ytr = build_dataset(tr_names, K)
Xva, Yva = build_dataset(va_names, K)
m_long = NameMLP(V, K, 8, 128)
opt = torch.optim.Adam(m_long.parameters(), lr=3e-3)
g = torch.Generator().manual_seed(SEED)

epochs_ax, val_ax, copied_ax = [], [], []
t0 = time.time()
for ep in range(1, 1201):
    idx = torch.randperm(len(Xtr), generator=g)
    for k in range(0, len(idx), 1024):
        b = idx[k:k + 1024]
        loss = F.cross_entropy(m_long(Xtr[b]), Ytr[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
    if ep % 100 == 0:                         # checkpoint: val loss + 50 fresh names
        s = sample_names(m_long, K, n=50, seed=SEED + ep)
        epochs_ax.append(ep)
        val_ax.append(evaluate(m_long, Xva, Yva))
        copied_ax.append(sum(x in tr_set for x in s) / 50)
        elapsed = time.time() - t0
        print(f"epoch {ep:>4}: val loss {val_ax[-1]:.3f}, copied {copied_ax[-1]:>4.0%}   "
              f"({elapsed:.0f} s elapsed, ~{elapsed / ep * (1200 - ep):.0f} s left)")

epoch  100: val loss 1.777, copied   2%   (6 s elapsed, ~69 s left)


epoch  200: val loss 2.065, copied   8%   (13 s elapsed, ~63 s left)


epoch  300: val loss 2.309, copied  14%   (19 s elapsed, ~56 s left)


epoch  400: val loss 2.529, copied  30%   (25 s elapsed, ~50 s left)


epoch  500: val loss 2.713, copied  24%   (32 s elapsed, ~44 s left)


...

epoch  800: val loss 3.036, copied  42%   (50 s elapsed, ~25 s left)


epoch  900: val loss 3.118, copied  30%   (56 s elapsed, ~19 s left)


epoch 1000: val loss 3.176, copied  34%   (60 s elapsed, ~12 s left)


epoch 1100: val loss 3.192, copied  36%   (64 s elapsed, ~6 s left)


epoch 1200: val loss 3.239, copied  44%   (66 s elapsed, ~0 s left)


In [43]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=["validation loss (lower is better)",
                                    "share of generated names copied from the training set"])
fig.add_scatter(x=epochs_ax, y=val_ax, mode="lines+markers", line=dict(color=BLUE, width=2),
                row=1, col=1)
fig.add_hline(y=uniform_loss, line=dict(color=GRAY, dash="dash", width=1), row=1, col=1,
              annotation_text="uniform guessing", annotation_position="bottom right")
fig.add_hline(y=bigram_loss, line=dict(color=GRAY, dash="dot", width=1), row=1, col=1,
              annotation_text="bigram", annotation_position="bottom right")
fig.add_scatter(x=epochs_ax, y=copied_ax, mode="lines+markers", line=dict(color=RED, width=2),
                row=2, col=1)
fig.update_yaxes(range=[1.5, 4.0], row=1, col=1)     # headroom above the uniform line
fig.update_yaxes(tickformat=".0%", range=[0, 1], row=2, col=1)
fig.update_xaxes(title_text="epoch", row=2, col=1)
fig.update_layout(title="Train longer: memorization up, generalization down", showlegend=False,
                  width=820, height=580, template="plotly_white")
fig.show()

print(f"copied share: {copied_ax[0]:.0%} (epoch 100)  ->  {copied_ax[-1]:.0%} (epoch 1200)")
print(f"val loss:     {val_ax[0]:.3f} (epoch 100)  ->  {val_ax[-1]:.3f} (epoch 1200)"
      f"   (bigram {bigram_loss:.3f}, uniform {uniform_loss:.3f})")

copied share: 2% (epoch 100)  ->  44% (epoch 1200)
val loss:     1.777 (epoch 100)  ->  3.239 (epoch 1200)   (bigram 1.948, uniform 3.664)


The trajectory is unambiguous. By epoch 200 the network is already **worse on new names than
the counting bigram** (2.065 vs 1.948), and by epoch 1200 its val loss (3.239) has handed
back most of its advantage over uniform guessing (3.664). Meanwhile the share of copied names
climbs from 2% to 44%. The copied curve is noisy - each checkpoint samples only 50 names -
so read its drift, not its wiggles. (Train even longer and such a model can end up *worse
than uniform guessing* on unseen names while reciting the seen ones perfectly. Not a stupid
model: a model that spent its capacity on the wrong job.)

This is the modern face of [43]'s overfitting lesson: in a generative model, overfitting *is*
memorization - the same failure that makes large language models occasionally recite their
training data (a real privacy issue). The cure is the same as [43]'s: **early stopping** at
the val minimum, which is exactly what `fit()` did.

## 13. Bonus: what did the embeddings learn?

Each character became a learned 8-number vector - a row of the embedding table from
section 4.1. The net was never told anything about phonetics; anything phonetic we find, it
inferred purely from which characters are statistically interchangeable inside 689 surnames.
Two ways to look:

**1. Nearest neighbours in the full 8-D space** - no projection, no distortion: for each
letter, which letters got the most similar vectors (cosine similarity)?

In [44]:
E = final_model.emb.weight.detach().numpy()               # (39, 8): one row per character
En = E / np.linalg.norm(E, axis=1, keepdims=True)
sim = En @ En.T                                           # cosine similarity, letter x letter

print("nearest neighbours in the full 8-D embedding space:")
for ch in "աեէիոօսնր":
    i = stoi[ch]
    nearest = np.argsort(-sim[i])[1:5]                    # [0] is the letter itself
    print(f"  {ch}:  " + "  ".join(itos[j] for j in nearest))

nearest neighbours in the full 8-D embedding space:
  ա:  ն  ի  զ  ո
  ե:  է  ք  ւ  ո
  է:  հ  ե  ճ  ւ
  ի:  ա  ձ  ց  ք
  ո:  գ  ք  ռ  ե
  օ:  և  ք  ս  ղ
  ս:  ծ  օ  և  մ
  ն:  ա  ժ  զ  շ
  ր:  ճ  լ  մ  դ


**2. A 2-D map with UMAP** ([36] - your own tool from ch10). UMAP builds a neighbour graph in
the 8-D space and lays it out in 2-D so that neighbours stay neighbours. With only 39 points
we shrink its neighbourhood (`n_neighbors=8` instead of the default 15) and use cosine
distance, the same similarity as the table above.

In [45]:
# ~2-3 min on the instructor laptop - almost all of it one-time compilation of UMAP's numba
# code (on import and on the first fit); fitting 39 points is nothing.
import warnings
from tqdm import TqdmWarning

with warnings.catch_warnings():            # umap imports tqdm, which warns that the notebook
    warnings.simplefilter("ignore", TqdmWarning)   # progress-bar widget is missing; we never use it
    import umap

xy = umap.UMAP(n_neighbors=8, min_dist=0.3, metric="cosine",
               random_state=SEED, n_jobs=1).fit_transform(E)

VOWELS = set("աեէիոօ")                  # (ը never occurs in the 689 surnames)
kind = ["end token" if itos[i] == "." else "vowel" if itos[i] in VOWELS else "consonant"
        for i in range(V)]

fig = go.Figure()
for group, color in [("vowel", RED), ("consonant", BLUE), ("end token", ORANGE)]:
    idx = [i for i in range(V) if kind[i] == group]
    fig.add_scatter(x=xy[idx, 0], y=xy[idx, 1], mode="markers+text", name=group,
                    text=[itos[i] for i in idx], textposition="top center",
                    textfont=dict(size=16), marker=dict(color=color, size=10),
                    hovertemplate="%{text}<extra></extra>")
fig.update_layout(title="Learned character embeddings, UMAP to 2-D",
                  xaxis_title="UMAP 1", yaxis_title="UMAP 2",
                  width=740, height=620, template="plotly_white")
fig.show()

Read both honestly:

- **ե and է** (both an "e" sound) found each other: է is ե's nearest letter in the full
  8-D space, ե is է's second, and on the map they sit side by side. The model was never told
  they sound alike; it noticed they are *used* alike.
- **ա and ն** are mutual nearest neighbours too, which no phonetics explains - most likely
  both vectors are shaped by the one overwhelming pattern, «-յան». Similar vectors mean "the
  model treats them alike", not "they sound alike".
- There is **no vowel/consonant split**: the other vowels (ա, ի, ո, օ) sit among
  consonants, and ո and օ (both "o") do not find each other in this run.

A finding, not a failure. Karpathy's *makemore*, trained on 32,000 English names, does see
the vowels a, e, i, o, u cluster in exactly this kind of plot. Representation quality is
bought with data - the same lesson the near-tie leaderboard taught, now seen from inside the
network. And a caution about the map itself: with only 39 points, UMAP's layout depends on
`n_neighbors` and the seed. The neighbour table is the ground truth; the map is the overview.

## 14. Bonus: steer it - names from your initials

Conditioning, the cheapest way: pre-load the context with characters of your choice and let
the net finish. (The grown-up version of this idea is called *prompting*.)

In [46]:
@torch.no_grad()
def finish(model, K, prefix, n=6, temp=1.0, seed=SEED):
    """Like sample_names, but the name starts with `prefix` instead of from scratch."""
    gen = torch.Generator().manual_seed(seed)
    out = []
    for _ in range(n):
        ctx = ([0] * K + [stoi[c] for c in prefix])[-K:]   # the last K letters of the prefix
        s = prefix
        while True:
            p = F.softmax(model(torch.tensor([ctx]))[0] / temp, dim=0)
            i = int(torch.multinomial(p, 1, generator=gen))
            if i == 0 or len(s) > 25:
                break
            s += itos[i]
            ctx = ctx[1:] + [i]
        out.append(s.capitalize())
    return out


for prefix in ["տ", "քո", "մաշ"]:          # try your own initials here
    print(f"{prefix}...  ->  " + ", ".join(finish(final_model, final_K, prefix)))

տ...  ->  Տեֆրաբուրյան, Տարդիսյան, Տարանյան, Տիրյան, Տուդչեկյան, Տիրյան
քո...  ->  Քոյինզարյան, Քոլաջյան, Քոնյան, Քոյան, Քոյամյան, Քոլումյան
մաշ...  ->  Մաշյինզարյան, Մաշիկյան, Մաշյան, Մաշիկյան, Մաշյան, Մաշյան


## 15. What you actually built

A character-level **language model**: embeddings -> MLP -> softmax over the next token,
trained with cross-entropy, scored against honest baselines, saved to a file and loaded back.
That is Bengio et al. (2003) - the direct ancestor of every modern LLM. What separates it
from ChatGPT is scale and one architectural idea: **much** more data, a **much** longer
context, and *attention* in place of our fixed $K$-window (a later chapter). The training
loop, the loss, the sampling, the temperature knob, the memorization risk - those you have
now seen for real, at surname scale.

And keep the wandb habit: from here on, every experiment in this course is a run with its
config recorded.

**Next:** convolutional networks - what happens when the *input* has shape worth
respecting.